# Unit 12 solutions: sequence-aware retrieval (what you read *last* matters)

These reference solutions drill the unit's three ideas on the **session log** (`sessions.csv.gz`) and read every number honestly. Every path the course built before this unit treats a reader's history as a **bag** of books; here the history has an **order**, and the question is whether the order carries signal the bag misses.

Three honest headlines frame the work. First, **recency** wins: querying Unit 4's item-item cosine with only the last few distinct books lifts hit@10 well above the bag, with a sweet spot at about five books (`LastKItemItemPath`). Second, a first-order **transition** table (`TransitionRetrievalPath`) also beats the bag, and the **order-shuffled control** -- the same books in a random order, shuffled for *both* learning and querying -- collapses it, which is the evidence that the lift is order and not just the set of books. Third, the tiny **SASRec** attention model *does* learn order (the milestone's CI-asserted control: 0.423 ordered vs 0.383 shuffled), but at this recipe it **trails** item-item CF and trails the simple order paths by more -- a counting model captures these short-range item-to-item effects directly.

All numbers are **on the session log**, graded on `val`, and are not comparable to Units 2-11. Only train positives ever enter a learned statistic or a stored history; the sealed `test` split is never touched, and the series table is used only as a diagnostic lens in Exercise 8. The notebook fits SASRec **exactly once** (Exercise 5, its own cell); every other path is numpy and fits in about a second.

In [1]:
import bookrec
import numpy as np
import pandas as pd
from bookrec import (
    ItemItemRetrievalPath,
    LastKItemItemPath,
    PopularityRetrievalPath,
    SasrecRetrievalPath,
    TransitionRetrievalPath,
    first_unseen_val,
    load_series,
    load_train_sequences,
    reader_hits,
    run_blended_scoreboard,
)
from bookrec.protocol import BaseRetrievalPath, Candidate

K = 10                           # every scoreboard reports at the top-10 cutoff
SHUFFLE_SEED = 20260930 + 1000   # the order-shuffled control's seed (the lesson's)

# --- the SESSION log (Unit 12 only): rows carry a timestamp, so each history has an order ---
data_dir = bookrec.generated_dir()
catalog_ids = sorted(bookrec.load_catalog(data_dir / "catalog.csv.gz"))   # 2000 books
col_of = {item_id: col for col, item_id in enumerate(catalog_ids)}      # book -> matrix column
sessions_path = data_dir / "sessions.csv.gz"
rows = pd.read_csv(sessions_path).to_dict("records")   # reader_id, item_id, timestamp, split, label, ...

# Each reader's ORDERED train-positive history (oldest first; val/test never enter it).
sequences = load_train_sequences(sessions_path)
# Each scored reader's time-ordered FIRST val positive that is not a re-read (the next-item target).
next_item = first_unseen_val(sessions_path)


def board(path):
    # hit@10 + catalog coverage on `val` via the single-path blended scoreboard (reproduces hit@k).
    result = run_blended_scoreboard({path.name: path}, sessions_path, catalog_ids=catalog_ids, k=K, pool=K)
    return result.hit_rate_at_k, result.catalog_coverage


# The two baselines every exercise compares against (both bag models, from Units 2 and 4).
popularity = PopularityRetrievalPath().fit(rows, catalog=catalog_ids)
item_item = ItemItemRetrievalPath().fit(rows, catalog=catalog_ids)
item_item_hit, item_item_cov = board(item_item)       # ~0.480 / ~0.336 on the session log
popularity_hit, popularity_cov = board(popularity)    # ~0.113 / ~0.008

## Exercise 1 - The last-k dial: from "remember everything" to "only the last book"

I fit `LastKItemItemPath` at `k_last` = 1, 3, 5 and `None`, board each on `val`, and read the dial. `None` queries with every distinct train book -- exactly Unit 4's bag query -- so it must reproduce item-item's hit@10 to the last digit; that is the check that the new path is the same cosine with only the query changed. Shortening the window to 3 or 5 recent books lifts hit@10 well above the bag, peaking at 5; a window of one book is too little context and falls below the bag.

In [2]:
k_values = [1, 3, 5, None]
last_k = {k: LastKItemItemPath(k).fit(rows, catalog=catalog_ids) for k in k_values}
sweep = {k: board(last_k[k]) for k in k_values}   # {k: (hit@10, coverage)}

none_is_item_item = bool(abs(sweep[None][0] - item_item_hit) < 1e-12)   # the bag end of the dial
recency_lifts = bool(sweep[3][0] > item_item_hit)
best_k = max(k_values, key=lambda k: sweep[k][0])
one_is_too_short = bool(sweep[1][0] < item_item_hit)

dial_explanation = (
    "A short window queries with the books that predict the NEXT read -- the series volume just "
    "finished, the current author streak or genre mood -- and drops tastes from months ago that no "
    "longer drive what the reader picks up. A window of one throws away the rest of that recent "
    "context, so a single (possibly off-mood) book steers everything; the sweet spot is a handful "
    "of recent distinct books."
)

print(f"item-item (bag)   hit@10={item_item_hit:.3f}  coverage={item_item_cov:.3f}")
print(f"popularity        hit@10={popularity_hit:.3f}  coverage={popularity_cov:.3f}")
for k in k_values:
    label = "all" if k is None else str(k)
    print(f"last-{label:<4}        hit@10={sweep[k][0]:.3f}  coverage={sweep[k][1]:.3f}")
print(f"None == item-item: {none_is_item_item} | recency lifts: {recency_lifts} | "
      f"best k: {best_k} | k=1 too short: {one_is_too_short}")
print(dial_explanation)

assert none_is_item_item, "k_last=None is the bag query: it must reproduce item-item's hit@10 exactly"
assert recency_lifts, "querying with the last 3 distinct books beats the bag"
assert best_k == 5, "the sweep peaks at a window of five recent books"
assert one_is_too_short, "a single recent book is too little context: it falls below the bag"


item-item (bag)   hit@10=0.480  coverage=0.336
popularity        hit@10=0.113  coverage=0.007
last-1           hit@10=0.362  coverage=0.807
last-3           hit@10=0.551  coverage=0.713
last-5           hit@10=0.572  coverage=0.606
last-all         hit@10=0.480  coverage=0.336
None == item-item: True | recency lifts: True | best k: 5 | k=1 too short: True
A short window queries with the books that predict the NEXT read -- the series volume just finished, the current author streak or genre mood -- and drops tastes from months ago that no longer drive what the reader picks up. A window of one throws away the rest of that recent context, so a single (possibly off-mood) book steers everything; the sweet spot is a handful of recent distinct books.


## Exercise 2 - Count item transitions by hand, then reveal `TransitionRetrievalPath`

I build the transition matrix $T$ by hand: walk every reader's **ordered train** history and add one count per consecutive pair $(a, b)$ at `T[col_of[a], col_of[b]]`. The total must equal the number of consecutive pairs, and the shipped path's `transitions` must equal mine element for element. Then I recompute one reader's score vector by hand -- the sum of the $T$ rows of their last 5 books -- and confirm every candidate the path retrieves for that reader carries exactly that score. Finally I board the path: it beats the bag (item-item).

In [3]:
T = np.zeros((len(catalog_ids), len(catalog_ids)))
for history in sequences.values():
    for a, b in zip(history[:-1], history[1:]):
        T[col_of[a], col_of[b]] += 1.0
n_consecutive_pairs = sum(len(history) - 1 for history in sequences.values())
one_count_per_pair = bool(int(T.sum()) == n_consecutive_pairs)

transition = TransitionRetrievalPath(5).fit(rows, catalog=catalog_ids)
matches_path = bool(np.array_equal(T, transition.transitions))

reader = min(r for r in sequences if len(sequences[r]) >= 6)
history = sequences[reader]                       # oldest first
hand_scores = T[[col_of[item] for item in history[-5:]]].sum(axis=0)   # one score per book
path_top = transition.retrieve(reader, {"seen": set(history)}, K)
scores_match = bool(path_top) and all(
    abs(hand_scores[col_of[c.item_id]] - c.score) < 1e-9 for c in path_top
)

transition_hit, transition_cov = board(transition)

leakage_explanation = (
    "T may count only consecutive TRAIN positives, because the step from a reader's last train book "
    "to their first val book is exactly the 'what comes next' the scoreboard asks the model to "
    "predict. Counting that pair (or any val/test read) would write the answer into the very "
    "statistic used to score it, so the measured hit@10 would be inflated by leakage, not skill."
)

print(f"T: {int(T.sum())} counts over {n_consecutive_pairs} consecutive train pairs "
      f"(one per pair: {one_count_per_pair}); equals TransitionRetrievalPath.transitions: {matches_path}")
print(f"reader {reader}: {len(history)} train books; last 5 = {history[-5:]}")
print("path top-10:", [(c.item_id, c.score) for c in path_top])
print(f"every retrieved score equals the by-hand T-row sum: {scores_match}")
print(f"transition        hit@10={transition_hit:.3f}  coverage={transition_cov:.3f}  "
      f"(item-item {item_item_hit:.3f})")
print(leakage_explanation)

assert one_count_per_pair, "T holds exactly one count per consecutive train pair"
assert matches_path, "the by-hand T equals the shipped path's transition matrix exactly"
assert scores_match, "the path's retrieved scores are the sum of T rows of the last 5 books"
assert transition_hit > item_item_hit, "the transition path beats the bag on the session log"


T: 40381 counts over 40381 consecutive train pairs (one per pair: True); equals TransitionRetrievalPath.transitions: True
reader 0: 11 train books; last 5 = [823, 1759, 1572, 1323, 1365]
path top-10: [(1609, 12.0), (1341, 10.0), (452, 9.0), (1614, 7.0), (1960, 7.0), (1768, 6.0), (484, 5.0), (95, 4.0), (350, 4.0), (569, 4.0)]
every retrieved score equals the by-hand T-row sum: True
transition        hit@10=0.516  coverage=0.369  (item-item 0.480)
T may count only consecutive TRAIN positives, because the step from a reader's last train book to their first val book is exactly the 'what comes next' the scoreboard asks the model to predict. Counting that pair (or any val/test read) would write the answer into the very statistic used to score it, so the measured hit@10 would be inflated by leakage, not skill.


## Exercise 3 - The order-shuffled control: is it the order, or just the set of books?

I fit the shuffled control -- every reader's history randomly permuted, used for **both** counting $T$ and choosing the last-5 query books -- and board it. The ordered path beats it by more than 1.3x and by more than +0.10, so the transition lift is order, not just co-occurrence. I read both paths' stored `histories` once each and confirm the control is a true permutation (same books per reader, at least one order changed).

My prediction for last-3 CF, written before running it: **shuffling will hurt it too.** The cosine itself is order-free, but *which* three books count as "last" is order information; after a shuffle the query is three random books from the history. The measurement confirms it: last-3 collapses toward (and below) the bag.

In [4]:
transition_shuffled = TransitionRetrievalPath(5, shuffle_seed=SHUFFLE_SEED).fit(rows, catalog=catalog_ids)
shuffled_hit, shuffled_cov = board(transition_shuffled)

order_ratio = transition_hit / shuffled_hit
order_evidence = bool(order_ratio >= 1.3 and transition_hit >= shuffled_hit + 0.10)

ordered_histories = transition.histories             # read ONCE (the property returns a full copy)
shuffled_histories = transition_shuffled.histories   # read ONCE
same_books = bool(
    ordered_histories.keys() == shuffled_histories.keys()
    and all(sorted(shuffled_histories[r]) == sorted(ordered_histories[r]) for r in ordered_histories)
)
order_changed = any(shuffled_histories[r] != ordered_histories[r] for r in ordered_histories)

last3_prediction = (
    "Shuffling will hurt last-3 CF as well: the cosine is order-free, but choosing WHICH three books "
    "are 'last' is order information, so a shuffled history queries with three random books."
)
last3_shuffled = LastKItemItemPath(3, shuffle_seed=SHUFFLE_SEED).fit(rows, catalog=catalog_ids)
last3_shuffled_hit = board(last3_shuffled)[0]
last3_drop = sweep[3][0] - last3_shuffled_hit

control_explanation = (
    "The control keeps every reader's SET of books and destroys only their ORDER, so the drop from "
    "ordered to shuffled is the evidence that order carries the lift. It must shuffle BOTH the "
    "learning histories (the counted pairs) and the query histories (the last-5 books); shuffling "
    "only one half leaves real order in the other and gives a weaker, misleading control. Last-3 CF "
    "collapses too, which shows recency gets its information from order as well: the similarity "
    "table is order-free, but knowing which books were read last is exactly the order signal."
)

print(f"transition ordered hit@10={transition_hit:.3f} | shuffled hit@10={shuffled_hit:.3f} "
      f"(coverage {shuffled_cov:.3f}) | ratio={order_ratio:.2f} | +{transition_hit - shuffled_hit:.3f}")
print(f"order evidence (>=1.3x and >=+0.10): {order_evidence}")
print(f"same books per reader: {same_books} | some order changed: {order_changed}")
print("prediction:", last3_prediction)
print(f"last-3 ordered hit@10={sweep[3][0]:.3f} | last-3 shuffled hit@10={last3_shuffled_hit:.3f} "
      f"(drop {last3_drop:.3f}; item-item {item_item_hit:.3f})")
print(control_explanation)

assert order_evidence, "the ordered transition path beats its shuffled control by >=1.3x and >=+0.10"
assert same_books and order_changed, "the control is a true permutation: same books, new order"
assert last3_shuffled_hit < sweep[3][0] - 0.05, "shuffling also collapses recency CF (order picks the last books)"


transition ordered hit@10=0.516 | shuffled hit@10=0.333 (coverage 0.351) | ratio=1.55 | +0.183
order evidence (>=1.3x and >=+0.10): True
same books per reader: True | some order changed: True
prediction: Shuffling will hurt last-3 CF as well: the cosine is order-free, but choosing WHICH three books are 'last' is order information, so a shuffled history queries with three random books.
last-3 ordered hit@10=0.551 | last-3 shuffled hit@10=0.347 (drop 0.204; item-item 0.480)
The control keeps every reader's SET of books and destroys only their ORDER, so the drop from ordered to shuffled is the evidence that order carries the lift. It must shuffle BOTH the learning histories (the counted pairs) and the query histories (the last-5 books); shuffling only one half leaves real order in the other and gives a weaker, misleading control. Last-3 CF collapses too, which shows recency gets its information from order as well: the similarity table is order-free, but knowing which books were read last 

## Exercise 4 - Next-item hit vs any-item hit

Next-item hit@10 asks for **the** book a reader read next (their time-ordered first non-re-read `val` positive), not any held-out book. I compute it for item-item, last-5 and transition over all 1,419 readers in `next_item`. Every path scores lower on next-item than on any-item -- the metric is stricter -- and the order-aware paths' advantage over the bag, read as a ratio, is larger on next-item than on any-item.

In [5]:
def next_item_hit(path):
    # Fraction of readers in `next_item` whose next book is in the path's top K (train history = seen).
    hits = []
    for reader_id, target in next_item.items():
        seen = set(sequences.get(reader_id, ()))
        top = bookrec.rank(path.retrieve(reader_id, {"seen": seen}, K), n=K, exclude=seen)
        hits.append(1.0 if target in {c.item_id for c in top} else 0.0)
    return float(np.mean(hits))


n_next_readers = len(next_item)
next_item_item = next_item_hit(item_item)
next_last5 = next_item_hit(last_k[5])
next_transition = next_item_hit(transition)

next_below_any = bool(
    next_item_item < item_item_hit and next_last5 < sweep[5][0] and next_transition < transition_hit
)
any_ratio_transition = transition_hit / item_item_hit
next_ratio_transition = next_transition / next_item_item
any_ratio_last5 = sweep[5][0] / item_item_hit
next_ratio_last5 = next_last5 / next_item_item
order_gains_more_on_next = bool(next_ratio_transition > any_ratio_transition)

next_item_explanation = (
    "Any-item hit forgives the model for finding ANY of a reader's held-out books, while next-item "
    "hit asks for exactly the one they read next, so it is the stricter metric and every path "
    "scores lower on it. The order-aware paths aim directly at 'what comes next' from the last few "
    "books, so their edge over the bag grows when the metric asks for the next book specifically."
)

print(f"next-item readers: {n_next_readers}")
print(f"item-item   next-item={next_item_item:.3f}  any-item={item_item_hit:.3f}")
print(f"last-5      next-item={next_last5:.3f}  any-item={sweep[5][0]:.3f}")
print(f"transition  next-item={next_transition:.3f}  any-item={transition_hit:.3f}")
print(f"next below any for every path: {next_below_any}")
print(f"transition / item-item: any-item x{any_ratio_transition:.2f} | next-item x{next_ratio_transition:.2f}")
print(f"last-5 / item-item:     any-item x{any_ratio_last5:.2f} | next-item x{next_ratio_last5:.2f}")
print(f"order gains more on next-item: {order_gains_more_on_next}")
print(next_item_explanation)

assert n_next_readers == 1419, "next_item covers every scored reader with a non-re-read val target"
assert next_below_any, "next-item hit is stricter: below any-item hit for every path"
assert order_gains_more_on_next, "the order path's relative gain over the bag is larger on next-item"
assert next_ratio_last5 > any_ratio_last5, "recency's relative gain over the bag is also larger on next-item"


next-item readers: 1419
item-item   next-item=0.245  any-item=0.480
last-5      next-item=0.388  any-item=0.572
transition  next-item=0.298  any-item=0.516
next below any for every path: True
transition / item-item: any-item x1.07 | next-item x1.22
last-5 / item-item:     any-item x1.19 | next-item x1.59
order gains more on next-item: True
Any-item hit forgives the model for finding ANY of a reader's held-out books, while next-item hit asks for exactly the one they read next, so it is the stricter metric and every path scores lower on it. The order-aware paths aim directly at 'what comes next' from the last few books, so their edge over the bag grows when the metric asks for the next book specifically.


## Exercise 5 - One SASRec fit, read honestly against item-item CF

I fit the tiny causal self-attention model **once**, in its own cell (about 45 s), and reuse it everywhere below. Then I compare it with item-item CF **reader by reader** with `reader_hits`, which scores exactly the scoreboard's cohort -- so each mean equals that path's scoreboard hit@10, a check I assert. The paired difference SASRec minus item-item is negative by several paired standard errors: SASRec **trails** the bag, and trails the order paths by more. The shuffled-SASRec control is not refit here; I cite the milestone's CI-asserted numbers (0.423 ordered vs 0.383 shuffled, +0.040 at 3.8 paired SE) as stated constants and never assert on them.

In [6]:
# The ONE SASRec fit in this notebook (~45 s). Do not refit it below -- reuse `sasrec`.
sasrec = SasrecRetrievalPath().fit(rows, catalog=catalog_ids)
print(f"SASRec fitted: {len(sasrec.loss_history)} epochs, loss {sasrec.loss_history[0]:.3f} -> {sasrec.loss_history[-1]:.3f}")


SASRec fitted: 30 epochs, loss 7.303 -> 5.058


In [7]:
sasrec_hit, sasrec_cov = board(sasrec)

sasrec_by_reader = reader_hits(sasrec, sessions_path, catalog_ids=catalog_ids, k=K)
item_item_by_reader = reader_hits(item_item, sessions_path, catalog_ids=catalog_ids, k=K)
same_cohort = bool(sasrec_by_reader.keys() == item_item_by_reader.keys())
diffs = np.array([sasrec_by_reader[r] - item_item_by_reader[r] for r in sorted(item_item_by_reader)])
gap_to_item_item = float(diffs.mean())
paired_se = float(np.std(diffs, ddof=1) / np.sqrt(len(diffs)))
trails_item_item = bool(gap_to_item_item < 0)
trails_order_paths = bool(sasrec_hit < transition_hit and sasrec_hit < sweep[3][0])

# reader_hits scores exactly the scoreboard's cohort, so its means reproduce the boarded hit@10s.
means_match_board = bool(
    abs(float(np.mean(list(sasrec_by_reader.values()))) - sasrec_hit) < 1e-12
    and abs(float(np.mean(list(item_item_by_reader.values()))) - item_item_hit) < 1e-12
)

# The ordered-vs-shuffled SASRec control is MEASURED by this unit's milestone (CI), not refit here.
CITED_SASREC_ORDERED, CITED_SASREC_SHUFFLED = 0.423, 0.383   # milestone: +0.040, 3.8 paired SE

sasrec_reading = (
    "The cited control (0.423 ordered vs 0.383 shuffled, +0.040 at 3.8 paired SE, asserted by the "
    "milestone in CI) shows SASRec DOES learn order. But my own paired gap shows it TRAILS item-item "
    "CF by about 0.06, several standard errors below zero, and trails the simple order paths "
    "(last-3, transition) by more -- at this recipe it neither wins nor ties. The order effects in "
    "this log are short-range and item-to-item (next volume, same author), which a counting model "
    "captures directly from a few thousand pairs, while a tiny attention model trained for 30 "
    "epochs cannot turn its extra flexibility into accuracy on this little data."
)

print(f"SASRec            hit@10={sasrec_hit:.3f}  coverage={sasrec_cov:.3f}")
print(f"paired readers: {len(diffs)} (same cohort: {same_cohort}; reader_hits means == board: {means_match_board})")
print(f"SASRec - item-item = {gap_to_item_item:+.3f}  (paired SE {paired_se:.3f}; "
      f"{gap_to_item_item / paired_se:+.1f} SE)")
print(f"trails item-item: {trails_item_item} | trails last-3 ({sweep[3][0]:.3f}) and transition "
      f"({transition_hit:.3f}): {trails_order_paths}")
print(f"cited milestone control: ordered {CITED_SASREC_ORDERED} vs shuffled {CITED_SASREC_SHUFFLED} "
      f"(+{CITED_SASREC_ORDERED - CITED_SASREC_SHUFFLED:.3f})")
print(sasrec_reading)

assert same_cohort, "reader_hits pairs the two paths over exactly the same readers"
assert means_match_board, "the mean of reader_hits equals the scoreboard hit@10 for each path"
assert trails_item_item and gap_to_item_item + 2 * paired_se < 0, "SASRec trails item-item by >2 paired SE"
assert trails_order_paths, "SASRec trails the simple order paths at this recipe"


SASRec            hit@10=0.423  coverage=0.284
paired readers: 1419 (same cohort: True; reader_hits means == board: True)
SASRec - item-item = -0.057  (paired SE 0.012; -4.7 SE)
trails item-item: True | trails last-3 (0.551) and transition (0.516): True
cited milestone control: ordered 0.423 vs shuffled 0.383 (+0.040)
The cited control (0.423 ordered vs 0.383 shuffled, +0.040 at 3.8 paired SE, asserted by the milestone in CI) shows SASRec DOES learn order. But my own paired gap shows it TRAILS item-item CF by about 0.06, several standard errors below zero, and trails the simple order paths (last-3, transition) by more -- at this recipe it neither wins nor ties. The order effects in this log are short-range and item-to-item (next volume, same author), which a counting model captures directly from a few thousand pairs, while a tiny attention model trained for 30 epochs cannot turn its extra flexibility into accuracy on this little data.


## Exercise 6 - Coverage per path: accuracy is not the whole scoreboard

I collect every path's hit@10 and catalog coverage -- all already computed above, nothing refit or re-boarded -- into one table sorted by hit@10. Every recency path covers far more of the catalog than item-item, and coverage rises as the window shrinks. Last-1 shows the catch: the widest coverage among the personalized paths, with hit@10 well below the bag.

In [8]:
coverage_table = pd.DataFrame([
    {"path": "popularity", "hit@10": popularity_hit, "coverage": popularity_cov},
    {"path": "item-item", "hit@10": item_item_hit, "coverage": item_item_cov},
    {"path": "last-1", "hit@10": sweep[1][0], "coverage": sweep[1][1]},
    {"path": "last-3", "hit@10": sweep[3][0], "coverage": sweep[3][1]},
    {"path": "last-5", "hit@10": sweep[5][0], "coverage": sweep[5][1]},
    {"path": "transition", "hit@10": transition_hit, "coverage": transition_cov},
    {"path": "transition-shuffled", "hit@10": shuffled_hit, "coverage": shuffled_cov},
    {"path": "sasrec", "hit@10": sasrec_hit, "coverage": sasrec_cov},
]).sort_values("hit@10", ascending=False).reset_index(drop=True)

recency_widens = all(sweep[k][1] > item_item_cov for k in (1, 3, 5))
coverage_rises = bool(sweep[1][1] > sweep[3][1] > sweep[5][1])

coverage_explanation = (
    "The bag query sums similarity over a long history, so every reader's scores drift toward the "
    "same well-connected books and the top-10s look alike. A few recent books point each reader at "
    "their own small neighbourhood, so the top-10s spread across far more of the catalog. Last-1 "
    "has the widest coverage of the recency paths but poor hit@10 -- coverage alone is not the goal."
)

print(coverage_table.round(3))
print(f"every recency path covers more than item-item: {recency_widens} | "
      f"coverage rises as the window shrinks: {coverage_rises}")
print(coverage_explanation)

assert len(coverage_table) == 8, "one row per path in this notebook"
assert recency_widens, "every recency window covers more of the catalog than the bag"
assert coverage_rises, "coverage rises as the window shrinks: last-1 > last-3 > last-5"


                  path  hit@10  coverage
0               last-5   0.572     0.606
1               last-3   0.551     0.712
2           transition   0.516     0.369
3            item-item   0.480     0.336
4               sasrec   0.423     0.284
5               last-1   0.362     0.807
6  transition-shuffled   0.333     0.352
7           popularity   0.113     0.008
every recency path covers more than item-item: True | coverage rises as the window shrinks: True
The bag query sums similarity over a long history, so every reader's scores drift toward the same well-connected books and the top-10s look alike. A few recent books point each reader at their own small neighbourhood, so the top-10s spread across far more of the catalog. Last-1 has the widest coverage of the recency paths but poor hit@10 -- coverage alone is not the goal.


## Exercise 7 - Challenge: decayed recency instead of a hard last-k cutoff

I score every book by a weighted sum of item-item cosine columns over the reader's **whole** distinct history, weighting the most recent distinct book 1, the next `rate`, then `rate ** 2`, and so on. The scaffold `ScoredPath` turns that score function into a scoreboard-ready path. The honest result: neither smooth decay beats the hard last-5 window here.

In [9]:
class ScoredPath(BaseRetrievalPath):
    # Given scaffold: wraps score_fn(reader_id) -> one score per book (catalog_ids order), or None.
    def __init__(self, name, score_fn):
        super().__init__(name=name, version="1", _fitted=True)
        self.score_fn = score_fn

    def retrieve(self, query, context, k):
        scores = self.score_fn(int(query))
        if scores is None:
            return []
        seen = set(context.get("seen", ()))
        candidates = [
            Candidate(item_id, float(scores[col]), self.name)
            for col, item_id in enumerate(catalog_ids)
            if scores[col] > 0 and item_id not in seen
        ]
        return self._finish(candidates, k)


S = last_k[5].similarity   # the item-item cosine, rows/columns in catalog_ids order


def decayed_scores(reader_id, rate):
    # Weighted sum of similarity columns: most recent distinct book weight 1, then rate, rate**2, ...
    history = sequences.get(reader_id)
    if not history:
        return None
    w = np.zeros(len(catalog_ids))
    taken = set()
    age = 0
    for book in reversed(history):
        if book in taken:
            continue            # a re-read counts once, at its most recent position
        taken.add(book)
        w[col_of[book]] = rate ** age
        age += 1
    return S @ w


decay_hits = {
    rate: bookrec.run_validation_scoreboard(
        ScoredPath(f"decayed-{rate}", lambda r, rate=rate: decayed_scores(r, rate)),
        sessions_path, catalog_ids=catalog_ids, k=K,
    ).hit_rate_at_k
    for rate in (0.5, 0.7)
}
decay_beats_last5 = bool(max(decay_hits.values()) > sweep[5][0])

decay_explanation = (
    "A smooth decay does not beat the hard last-5 window here; the gentler rate only about ties it. "
    "The useful signal is short-range -- the last handful of books -- so keeping a long tail of "
    "older books, even at small weights, adds stale taste rather than information about the next read."
)

for rate, hit in decay_hits.items():
    print(f"decayed rate={rate}: hit@10={hit:.3f}")
print(f"hard last-5 hit@10={sweep[5][0]:.3f} | decay beats last-5: {decay_beats_last5}")
print(decay_explanation)

assert not decay_beats_last5, "recorded honestly: neither decay rate beats the hard last-5 window"
assert all(hit > item_item_hit for hit in decay_hits.values()), "both decays still beat the bag"


decayed rate=0.5: hit@10=0.510
decayed rate=0.7: hit@10=0.570
hard last-5 hit@10=0.572 | decay beats last-5: False
A smooth decay does not beat the hard last-5 window here; the gentler rate only about ties it. The useful signal is short-range -- the last handful of books -- so keeping a long tail of older books, even at small weights, adds stale taste rather than information about the next read.


## Exercise 8 - Challenge: the next-in-series diagnostic (series as a lens, never a feature)

I load the series table **only to select events**: scored readers whose last train book is volume $v$ of a series and whose next-item target is volume $v + 1$ of the same series. No path reads the table. On exactly those readers the spread is wide: the bag misses most next volumes, the transition path catches about half, and last-5 catches most -- learned from reading order alone.

In [10]:
series = load_series(data_dir / "series.csv.gz")   # diagnostics only -- never a model input

series_events = sorted(
    r for r in next_item
    if sequences.get(r)
    and sequences[r][-1] in series and next_item[r] in series
    and series[next_item[r]][0] == series[sequences[r][-1]][0]
    and series[next_item[r]][1] == series[sequences[r][-1]][1] + 1
)
n_series_events = len(series_events)


def series_hit(path):
    # Fraction of series_events whose next volume is in the path's top K (train history = seen).
    hits = []
    for reader_id in series_events:
        seen = set(sequences.get(reader_id, ()))
        top = bookrec.rank(path.retrieve(reader_id, {"seen": seen}, K), n=K, exclude=seen)
        hits.append(1.0 if next_item[reader_id] in {c.item_id for c in top} else 0.0)
    return float(np.mean(hits))


series_item_item = series_hit(item_item)
series_transition = series_hit(transition)
series_last5 = series_hit(last_k[5])
order_catches_series = bool(series_transition > series_item_item and series_last5 > series_item_item)

series_explanation = (
    "The bag query dilutes the just-finished volume among every older book in the history, so the "
    "next volume often loses to books similar to the reader's long-run taste. Recency puts the "
    "just-finished volume at the front of a short query, and 'volume v -> v+1' is a consecutive "
    "pair many other readers made, so both order paths catch it -- learned from reading order "
    "alone, never from the series table. Feeding `series` to a model would test a different idea "
    "(a hand-made content feature), and here it is forbidden: it is used only to choose events."
)

print(f"next-in-series events: {n_series_events}")
print(f"item-item  series hit@10={series_item_item:.3f}")
print(f"transition series hit@10={series_transition:.3f}")
print(f"last-5     series hit@10={series_last5:.3f}")
print(f"order paths catch next volumes better than the bag: {order_catches_series}")
print(series_explanation)

assert 100 <= n_series_events <= 200, "roughly 150 next-in-series events"
assert order_catches_series, "both order-aware paths catch next volumes better than the bag"


next-in-series events: 155
item-item  series hit@10=0.303
transition series hit@10=0.510
last-5     series hit@10=0.742
order paths catch next volumes better than the bag: True
The bag query dilutes the just-finished volume among every older book in the history, so the next volume often loses to books similar to the reader's long-run taste. Recency puts the just-finished volume at the front of a short query, and 'volume v -> v+1' is a consecutive pair many other readers made, so both order paths catch it -- learned from reading order alone, never from the series table. Feeding `series` to a model would test a different idea (a hand-made content feature), and here it is forbidden: it is used only to choose events.


## Exercise 9 - Challenge: blend recency CF with the transition path

I blend last-5 and transition with equal weights through Unit 6's blended scoreboard (pool 50) and read hit@10 and coverage against both single paths. The lift over the best single path (last-5) is small and the coverage lands between the two -- recorded honestly.

In [11]:
blend_result = run_blended_scoreboard(
    {"last-5": last_k[5], "transition": transition},
    sessions_path, catalog_ids=catalog_ids, k=K, pool=50,
)
blend_hit, blend_cov = blend_result.hit_rate_at_k, blend_result.catalog_coverage

best_single_hit = max(sweep[5][0], transition_hit)
blend_lift = blend_hit - best_single_hit
blend_beats_best = bool(blend_hit > best_single_hit)

blend_explanation = (
    "The lift is small because the two paths overlap heavily -- both key on the last few books. "
    "Similarity adds 'books like these' (same author, same genre neighbourhood) while transitions "
    "add 'what comes right after' (the next volume, the usual follow-up), so a second path is worth "
    "its cost only if a lift this small survives the noise of another seed or a fresh holdout."
)

print(f"last-5     hit@10={sweep[5][0]:.3f}  coverage={sweep[5][1]:.3f}")
print(f"transition hit@10={transition_hit:.3f}  coverage={transition_cov:.3f}")
print(f"blend      hit@10={blend_hit:.3f}  coverage={blend_cov:.3f}")
print(f"lift over best single path: {blend_lift:+.3f} | blend beats best: {blend_beats_best}")
print(blend_explanation)

assert abs(blend_lift) < 0.05, "blending two overlapping order paths moves hit@10 only a little"
assert min(sweep[5][1], transition_cov) <= blend_cov <= max(sweep[5][1], transition_cov) + 0.05, \
    "the blend's coverage lands between (or near) the two single paths"


last-5     hit@10=0.572  coverage=0.606
transition hit@10=0.516  coverage=0.369
blend      hit@10=0.579  coverage=0.498
lift over best single path: +0.008 | blend beats best: True
The lift is small because the two paths overlap heavily -- both key on the last few books. Similarity adds 'books like these' (same author, same genre neighbourhood) while transitions add 'what comes right after' (the next volume, the usual follow-up), so a second path is worth its cost only if a lift this small survives the noise of another seed or a fresh holdout.
